# Length-Aware Category Batching — Proxy Benchmark (Colab T4×2)

Tests the data pipeline **without real training**. It estimates one-epoch training
time via a **proxy** (cost ∝ padded tokens the GPU would process) and compares the
**round-robin category scheduler** against a length-agnostic **baseline**.

**Categories:** CAT1 = Short 40% / Medium 60% · CAT2 = Medium 45% / Long 55% · CAT3 = Chunked.
One category per batch, round-robin across batches; empty-band → fill from the other
band (signal a refill unless the resident shards are exhausted).

Runs single-process **and** distributed (`torchrun --nproc_per_node=2`) — the DDP
machinery still runs, it just does the proxy instead of an optimizer step.
UDS is kept separate and is **not** used here.

## 0 · Clean clone & locate
Uploads or clones the project, finds the folder with `bindings.cpp`.

In [ ]:
import os, sys, glob, shutil, subprocess

# Option A: clone your repo (edit the URL). Option B: upload sft_final.zip and unzip.
REPO_URL = "https://github.com/SniAssia/sft.git"
if not glob.glob("/content/**/bindings.cpp", recursive=True):
    shutil.rmtree("/content/sft", ignore_errors=True)
    try:
        subprocess.run(["git","clone","--depth","1",REPO_URL,"/content/sft"], check=True)
    except Exception as e:
        print("clone failed:", e, "\n-> upload sft_final.zip via the Files panel and unzip:")
        # from google.colab import files; up=files.upload()  # then: !unzip -q sft_final.zip -d /content/sft

def find_code_dir():
    for root, dirs, files in os.walk("/content"):
        dirs[:] = [d for d in dirs if d != ".git"]
        if "bindings.cpp" in files: return root
    return None

CODE = find_code_dir()
assert CODE, "bindings.cpp not found — clone or upload the project first"
sys.path.insert(0, CODE)
print("code dir:", CODE)

In [ ]:
import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0), torch.cuda.get_device_capability(0),
          "| count:", torch.cuda.device_count())
# transformers only needed to build shards with a real tokenizer
import importlib, subprocess, sys
if importlib.util.find_spec("transformers") is None:
    subprocess.run([sys.executable,"-m","pip","install","-q","transformers"], check=True)

## 1 · Build the C++ extension

In [ ]:
b = subprocess.run([sys.executable,"setup.py","build_ext","--inplace"],
                   cwd=CODE, capture_output=True, text=True)
print(b.stdout[-800:])
if b.returncode!=0: print("STDERR:\n",b.stderr[-2500:]); raise RuntimeError("build failed")
import uds_loader
print("uds_loader OK:", [x for x in dir(uds_loader) if not x.startswith("_")])

## 2 · Build shards

Use **synthetic** shards (instant, no downloads) to exercise the pipeline, OR real
data via `prepare_shards.py`. Synthetic is enough to compare batching methods.

In [ ]:
import random, prepare_shards as ps
SH = os.path.join(CODE, "_shards"); MAXLEN = 2048
cfg = ps.Config(out_dir=SH, max_seq_length=MAXLEN, shard_size=500,
                band_short_max=512, band_medium_max=1536)
w = ps.ShardWriter(cfg); random.seed(0)
for _ in range(6000):
    r = random.random()
    t = (random.randint(20,500) if r<.4 else random.randint(512,1500) if r<.7
         else random.randint(1536,2048) if r<.9 else random.randint(2100,4000))
    pl,cl = max(1,t//5), t//5; rl = t-pl-cl
    mk = lambda n: [random.randint(1,84000) for _ in range(n)]
    c,isk,bd = ps.decide_case(pl,cl,rl,MAXLEN); w.add(ps.TokSample(mk(pl),mk(cl),mk(rl),isk,c,bd))
w.flush()
print("bands S/M/L/C:", w.band_counts, "shards:", w.shard_idx)

# write a meta.json (prepare_shards.run writes one for real data; synthetic path needs it)
import json
json.dump({"pad_id":0,"vocab_size":84000,"max_seq_length":MAXLEN,"option_b_window":MAXLEN,
           "num_samples":w.total_written,"band_counts":w.band_counts},
          open(os.path.join(SH,"meta.json"),"w"))
print("wrote meta.json")

### (optional) real data instead of synthetic
```python
!cd "{CODE}" && python prepare_shards.py --config datasets_real.json --out ./_shards \
    --tokenizer facebook/opt-125m --max-seq-length 1024 --shard-size 1024 --workers 4
```

## 3 · Proxy benchmark — single process (ours vs baseline)

In [ ]:
import importlib, proxy_benchmark, run_proxy
importlib.reload(proxy_benchmark); importlib.reload(run_proxy)
from proxy_benchmark import run_epoch_proxy, compare, ProxyModel

MAXLEN = 2048; B = 32; WINDOW = 4
meta = __import__("json").load(open(os.path.join(SH,"meta.json")))
model = ProxyModel(alpha=1.0, beta=0.0)     # relative padded-token units

def make(baseline):
    pc = uds_loader.PipelineConfig()
    pc.shards = sorted(glob.glob(os.path.join(SH,"shard_*.bin")))
    pc.num_epochs = 1; pc.resident_window = WINDOW; pc.B = B
    pc.baseline = baseline; pc.pad_id = int(meta["pad_id"])
    pc.option_b_window = MAXLEN; pc.pad_to_multiple = 8
    pc.prefetch_workers = 3; pc.ring_capacity = 4
    if not baseline:
        pc.profile_bands = [[0,1],[1,2],[3,3]]
        pc.profile_mix   = [[0.40,0.60],[0.45,0.55],[1.0,0.0]]
        pc.profile_is_chunked = [False,False,True]
    p = uds_loader.DataPipeline(pc); p._baseline = baseline; return p

p = make(False); p.start(); ours = run_epoch_proxy(p, model); p.stop(); ours.method="round_robin"
p = make(True);  p.start(); base = run_epoch_proxy(p, model); p.stop(); base.method="baseline"
print(compare(ours, base))

In [ ]:
# full per-category / health detail
import json
print(json.dumps(ours.summary(), indent=2))

## 4 · Distributed proxy — T4×2 (`torchrun --nproc_per_node=2`)

The DDP system runs for real; each rank runs the proxy over its shard subset and
totals are all-reduced. No optimizer step, no model.

In [ ]:
# needs 2 GPUs: Runtime > Change runtime type > T4 x2 (or run with 1)
NPROC = min(2, max(1, torch.cuda.device_count()))
print("launching torchrun with nproc_per_node =", NPROC)
r = subprocess.run(
    ["torchrun", f"--nproc_per_node={NPROC}", "run_proxy.py",
     "--shards", SH, "--B", "32", "--window", "4", "--alpha", "1.0"],
    cwd=CODE, capture_output=True, text=True)
print(r.stdout[-3000:])
if r.returncode != 0: print("STDERR:\n", r.stderr[-3000:])

In [ ]:
# the distributed run saved proxy_benchmark.json in CODE
import json, os
pth = os.path.join(CODE, "proxy_benchmark.json")
if os.path.exists(pth): print(json.dumps(json.load(open(pth)), indent=2))

## 5 · Notes
- **Proxy time** is in relative padded-token units (`alpha=1`). To get seconds, fit
  `alpha`/`beta` from a few timed real steps and pass them to `ProxyModel`.
- **Resident window `W`** (`--window`) trades RAM for smoother mixing / fewer fallbacks.
- **UDS** (`uds.py`) is untouched and not used in this benchmark.